# Rust 104: Strings

Chapter 04 of the Rust track. Chapters 01–03 covered values, control flow and functions. This chapter is about **text**: `String` versus `&str`, the fact that Rust strings are UTF-8 byte buffers rather than arrays of characters, why indexing is deliberately absent, and the method inventory you will actually use.

## Who this is written for
You are assumed to be a competent programmer, arriving from Python and C++. Python taught you that a string is a sequence of code points and that slicing is always safe. C++ taught you that a string is a byte buffer and that slicing is always your problem. Rust is the third thing: a **UTF-8 byte buffer with an API that refuses to let you pretend bytes are characters.**

## How to read this notebook
- Markdown cells explain; code cells demonstrate. Run each code cell in order.
- Code that intentionally fails is shown as a comment with the compiler's actual diagnostic. Reading those diagnostics is part of the material.
- The prose is terse and factual: this is a reference, not a novel.

## The one idea to internalise

A Rust `String` is a **`Vec<u8>` that has been validated as UTF-8**. `&str` is a **borrowed view into such a buffer**, with a length in bytes. Everything else follows from that single fact:

| Consequence | Why |
|---|---|
| `.len()` is bytes, not characters | It is a byte-buffer length |
| You cannot write `s[0]` | Byte 0 is not necessarily a character |
| Slicing can panic at runtime | The compiler cannot prove a byte offset is a char boundary |
| `.chars()` is an *iterator*, not an index | Walking characters is O(n) |
| `"héllo".len() == 6` but `"héllo".chars().count() == 5` | `é` is two bytes |

The strictness is a trade. You lose the convenience of `s[i]`, and in exchange you get text handling that cannot silently corrupt data, and APIs that make allocation visible.


### Contents
- [0. Notebook Conventions: How These Cells Run](#0-notebook-conventions-how-these-cells-run)
- [1. `String` and `&str`: Two Types, One Buffer](#1-string-and-str-two-types-one-buffer)
- [2. UTF-8: Bytes, Chars and Boundaries](#2-utf-8-bytes-chars-and-boundaries)
- [3. Slicing, and Why It Can Panic](#3-slicing-and-why-it-can-panic)
- [4. Literals: Escapes, Raw Strings, Multi-Line](#4-literals-escapes-raw-strings-multi-line)
- [5. Growing and Shrinking a `String`](#5-growing-and-shrinking-a-string)
- [6. Iterating: `bytes`, `chars`, `char_indices`](#6-iterating-bytes-chars-char_indices)
- [7. The Method Inventory](#7-the-method-inventory)
- [8. Splitting, Lines and Joining](#8-splitting-lines-and-joining)
- [9. Converting Between Bytes and Text](#9-converting-between-bytes-and-text)
- [10. Passing Strings: `&str` in, `String` out](#10-passing-strings-str-in-string-out)
- [11. Text Processing Patterns](#11-text-processing-patterns)
- [12. Formatting and Display](#12-formatting-and-display)
- [13. Reading the Compiler on Strings](#13-reading-the-compiler-on-strings)
- [14. Unicode Beyond ASCII](#14-unicode-beyond-ascii)
- [15. Performance Notes for Low-Latency Code](#15-performance-notes-for-low-latency-code)
- [16. Cheat Sheet](#16-cheat-sheet)

If you are returning to this chapter later, §16 is the fastest route back to the method names; §1 and §2 are the ones to re-read when something behaves oddly.


## 0. Notebook Conventions: How These Cells Run

This notebook runs on the **evcxr** kernel, which wraps each cell into a generated program and compiles it incrementally. Three consequences matter for every cell below:

- **There is no `fn main`.** The kernel supplies one; you are typing its body.
- **State persists across cells.** A `let` from cell 4 is still live in cell 30.
- **Items are order-independent, statements are not.** Two `fn`s may call each other regardless of order; two `let`s may not use each other's names before they run.

There is **one namespace per notebook**. Two items with the same name collide exactly as they would in one source file, even in different sections. If you re-run a cell that defines an item, it is redefined; on a redefinition error, restart the kernel and run from the top.

A failing cell does not poison the kernel — an intentional panic in a later section is safe to run. `println!` writes to stdout; `eprintln!` and `dbg!` write to stderr.

### A habit worth forming

Strings produce more surprising runtime behaviour than any other primitive, because UTF-8's byte layout is invisible in the source. When a string operation misbehaves, print `.len()`, `.chars().count()` and `.as_bytes()` together before assuming the method is wrong. The bug is usually an assumption that bytes and characters are the same thing.


In [ ]:
// 0.1 Smoke test: a string literal, its byte length and its character count.
let text = "héllo";
println!("{text} — {} bytes, {} chars", text.len(), text.chars().count());


## 1. `String` and `&str`: Two Types, One Buffer

A string **literal** is `&'static str`: a pointer into the program's read-only data, plus a byte length. It is borrowed, immutable, and never freed because it is never owned. An owned **`String`** is a growable buffer on the heap — a `Vec<u8>` with an invariant: its contents are always valid UTF-8.

The relationship is the same one as `Vec<T>` and `&[T]`, or an `i32` and a `&i32`. One owns, one borrows. Neither is "the string type"; they are two views of the same layout.

| Type | Owns | Mutable | Where it lives | Typical role |
|---|---|---|---|---|
| `&str` | No | No | Anywhere (often `.rodata`) | Read-only parameters and views |
| `String` | Yes | Yes | Heap (ptr + len + capacity) | Built, grown, returned |
| `&String` | No | No | Borrow of a `String` | Rare — prefer `&str` |

The size difference is the giveaway: `&str` is 16 bytes on a 64-bit target (pointer + length), while `String` is 24 (pointer + length + capacity), exactly like a `Vec`.


In [ ]:
// 1.1 A literal is a borrowed, 'static view into the binary's read-only data.
let literal: &'static str = "immutable text";
println!("{literal} — {} bytes, never freed, never owned", literal.len());


### An owned buffer

`String` is created in three common ways:

| Expression | Allocates | Notes |
|---|---|---|
| `String::new()` | No | Capacity 0; the first push allocates |
| `String::from("x")` | Yes | Copies the literal onto the heap |
| `"x".to_string()` | Yes | Same, via the `ToString` trait |

`String::new()` not allocating is worth remembering: it is the cheap default for an accumulator that may never be filled.


In [ ]:
// 1.2 Three ways to get an owned String, and the capacity each one starts with.
let from_new = String::new();
let from_str = String::from("mutable text");
let from_to_string = "mutable text".to_string();
println!("new: len={} cap={}", from_new.len(), from_new.capacity());
println!("from: {from_str} cap={}", from_str.capacity());
println!("to_string: {from_to_string} cap={}", from_to_string.capacity());


### `&String` and `&str` are not interchangeable *as types*

They are interchangeable *at call sites*, because of **deref coercion**: `&String` automatically becomes `&str` when a `&str` is expected. That is a compiler convenience, not a type identity — and the distinction matters when you write generic code or store references in a struct.

Comparison, however, works in both directions, because the standard library provides the `PartialEq` implementations:

```rust
"same" == String::from("same")   // true
```

The comparison is byte-wise, so it agrees with ordering (`"a" < "b"`) and with using either type as a `HashMap` key.


In [ ]:
// 1.3 Comparison crosses the &str/String boundary; the types themselves do not.
let borrowed: &str = "same";
let owned: String = String::from("same");
println!("&str == String : {}", borrowed == owned);
println!("String == &str : {}", owned == borrowed);
println!("byte-wise equal: {}", borrowed.as_bytes() == owned.as_bytes());


### The layout, in one line

For ASCII text the mental model "byte buffer" is exactly right and costs nothing:

```text
"hello"  →  [ h ][ e ][ l ][ l ][ o ]   ptr, len = 5
```

Each ASCII character is one byte, so `len()`, indexing arithmetic and iteration all agree. The moment a non-ASCII character appears, they diverge — which is the entire subject of §2.

One difference from C is worth stating plainly: Rust strings are **not NUL-terminated**. `"abc".len()` is 3, not 4. The `\0` you append for C interop is just a byte you added, and `.len()` counts it.


In [ ]:
// 1.4 For ASCII, bytes and characters coincide — which is why the bug hides.
let ascii = "hello";
println!("{:?} — len {} == chars {}", ascii.as_bytes(), ascii.len(), ascii.chars().count());
println!("not NUL-terminated: \"abc\\0\".len() == {}", "abc\0".len());


### Indexing does not exist, and that is a decision

The single most surprising thing about Rust strings for a newcomer is this:

```rust
let s = String::from("hi");
let c = s[0];        // does not compile
```

The compiler's message is unusually helpful, and worth reading once carefully:

```text
error[E0277]: the type `str` cannot be indexed by `{integer}`
   |
   |     let c = s[0];
   |                ^ string indices are ranges of `usize`
   |
   = note: you can use `.chars().nth()` or `.bytes().nth()`
```

Two facts are encoded here. First, the `Index` trait for `str` is implemented for **ranges of `usize`**, not for `usize` — so `&s[0..1]` is legal syntax. Second, the reason it is not implemented for a single index is that a single byte is not a character, and the language refuses to hand you a `char` that may not exist.


In [ ]:
// 1.5 Integer indexing is a compile error; a byte range is the legal spelling.
//
//     let s = String::from("hi");
//     let c = s[0];
//
// error[E0277]: the type `str` cannot be indexed by `{integer}`
//   = note: you can use `.chars().nth()` or `.bytes().nth()`
//
// What does work — a byte range that happens to land on a char boundary:
let s = String::from("hi");
println!("&s[0..1] = {:?}", &s[0..1]);


## 2. UTF-8: Bytes, Chars and Boundaries

Rust source files, `String`, and `&str` are all **UTF-8**. Not UTF-16, not a code-point array, not platform-dependent. That choice is fixed at the type level, which is why the same code produces identical byte sequences on Windows and Linux.

A UTF-8 encoded **scalar value** is 1 to 4 bytes:

| Range | Bytes | Example |
|---|---|---|
| U+0000–U+007F | 1 | `a`, `7`, `\n` |
| U+0080–U+07FF | 2 | `é`, `ß`, `α` |
| U+0800–U+FFFF | 3 | `€`, `中`, `€` |
| U+10000–U+10FFFF | 4 | `👋`, `𝄞` |

A **char boundary** is a byte index at which a character starts. Byte index 0 is always a boundary; byte index `len()` is always a boundary; everything else depends on the text.


In [ ]:
// 2.1 len() counts bytes; chars().count() counts scalar values. They differ.
let text = "héllo";
println!("{text}: len = {}, chars = {}", text.len(), text.chars().count());
println!("bytes = {:?}", text.as_bytes());


### The size of a character, in bytes

`char::len_utf8()` reports the encoded width, and `char::encode_utf8()` writes it into a buffer. Both make the byte cost of a character explicit rather than implied.

Note the naming: `len_utf8` exists precisely because `len` alone would be ambiguous. The standard library is careful about this, and you should be too when you write APIs.


In [ ]:
// 2.2 How wide is each character? The answer is not always one byte.
for c in "aé€👋".chars() {
    println!("{c:?} — {} byte(s), U+{:04X}", c.len_utf8(), c as u32);
}


### Where the characters are: `char_indices`

`char_indices()` yields `(byte_index, char)` pairs. It is the tool to reach for whenever you need to *know* a position rather than just visit a character — and the only correct way to build a byte index for slicing.

`is_char_boundary(i)` answers the question directly: may I slice here?


In [ ]:
// 2.3 char_indices gives byte offsets; is_char_boundary tells you where you may slice.
let text = "aé";
for (index, c) in text.char_indices() {
    println!("byte {index} -> {c:?}");
}
let hello = "héllo";
println!("boundaries: 1 -> {}, 2 -> {}, 3 -> {}",
    hello.is_char_boundary(1), hello.is_char_boundary(2), hello.is_char_boundary(3));


### Iteration is not indexing

Because the encoding is variable-width, the *n*-th character cannot be found without walking. There is no O(1) `chars()` index. This is the same trade-off Python's `str` makes — Python's `s[i]` is also O(n) for non-ASCII — except Rust makes the cost visible by not giving you the operator at all.

If you truly need random access to characters, collect them:

```rust
let chars: Vec<char> = text.chars().collect();   // O(n) once, O(1) after
```

That is usually the wrong trade for a hot path (4 bytes per ASCII character), and the right trade for a one-off parse.


In [ ]:
// 2.4 nth() walks; there is no constant-time character index.
let text = "héllo";
println!("chars().nth(1) = {:?}", text.chars().nth(1));
println!("bytes().nth(1) = {:?}", text.bytes().nth(1));
println!("collected chars = {:?}", text.chars().collect::<Vec<char>>());


### Recap: which length is which

| Call | Counts | Cost | Value for `"héllo"` |
|---|---|---|---|
| `.len()` | bytes | O(1) | 6 |
| `.chars().count()` | scalar values | O(n) | 5 |
| `.bytes().count()` | bytes | O(n) | 6 |
| `.encode_utf16().count()` | UTF-16 units | O(n) | 5 |

`.len()` is the only one that is free, because it is stored in the `&str` fat pointer. Everything else walks. In allocation-sensitive code, prefer APIs that work in bytes.


## 3. Slicing, and Why It Can Panic

`&s[a..b]` slices by **byte offsets**, and panics at runtime if `a` or `b` is not a char boundary:

```text
thread 'main' panicked at ...:
byte index 2 is not a char boundary; it is inside 'é' (bytes 1..3 of string)
```

Read that message closely: it names the character, and it gives the byte range the character occupies. The compiler cannot catch this, because byte offsets are runtime values; the panic is the safety net.

This is a real difference from both languages you know. Python would return a `str` (it slices by code point, and would never be "inside" a character). C++ would silently hand you half a UTF-8 sequence. Rust refuses both.


In [ ]:
// 3.1 Slicing by byte offset works when the offsets are char boundaries.
let text = "héllo";
println!("[..1] = {:?}", &text[..1]);
println!("[3..] = {:?}", &text[3..]);
println!("[..=0] = {:?}", &text[..=0]);


### The panic, demonstrated

A panic in a notebook cell is safe: the kernel survives, and the next cell still runs. The cell below catches the panic so the notebook continues cleanly, and prints the message the runtime produced.

The trick is `std::panic::catch_unwind`, which is how you observe a panic without ending the process. The default panic hook is silenced for the duration so the output stays readable — in a normal program you would leave it alone.


In [ ]:
// 3.2 A non-boundary slice panics; catch_unwind lets us print the message.
let previous = std::panic::take_hook();
std::panic::set_hook(Box::new(|_| {}));
let outcome = std::panic::catch_unwind(|| {
    let text = "héllo";
    let _ = &text[0..2]; // byte 2 is inside 'é' (bytes 1..3)
});
std::panic::set_hook(previous);
println!("panicked = {}", outcome.is_err());
if let Err(payload) = outcome {
    let message = payload.downcast_ref::<&str>().map(|s| s.to_string())
        .or_else(|| payload.downcast_ref::<String>().cloned());
    println!("message: {}", message.unwrap_or_default());
}


### The non-panicking alternative: `get`

`str::get(range)` returns `Option<&str>` instead of panicking. This is the idiom for parsing untrusted input, where a malformed offset should be handled rather than crash the process.

The same pattern appears on slices (`slice::get`), on `Vec` and on `HashMap`: a `get` that returns `Option`, and an index operator that panics. Choosing between them is choosing whether the failure is recoverable.

| Form | Failure mode | Use when |
|---|---|---|
| `&s[a..b]` | Panics | The range is provably valid |
| `s.get(a..b)` | `Option<&str>` | The range comes from input |
| `s.get(a..b).unwrap_or("")` | Empty on failure | A safe default is acceptable |


In [ ]:
// 3.3 get() returns Option instead of panicking — the input-facing form.
let text = "héllo";
println!("get(0..1) = {:?}", text.get(0..1));
println!("get(0..2) = {:?}", text.get(0..2));
println!("get(..2).unwrap_or(\"\") = {:?}", text.get(..2).unwrap_or(""));


### A slice is a borrow, and borrows have rules

A `&str` produced by slicing keeps the `String` immutably borrowed for as long as the slice lives. Mutating the `String` while a slice is alive is an error:

```text
error[E0502]: cannot borrow `s` as mutable because it is also borrowed as immutable
```

This is the borrow checker (chapter 05) showing up early, and it is exactly the rule that prevents the slice from being invalidated by a reallocation. The fix is to end the slice's life — copy what you need with `.to_string()`, or finish using the slice before mutating.


In [ ]:
// 3.4 A live slice blocks mutation of its owner.
//
//     let mut s = String::from("hello");
//     let slice = &s[0..2];
//     s.push('!');               // E0502: cannot borrow `s` as mutable
//     println!("{}", slice);     // ...because it is also borrowed as immutable
//
// Copying the slice out ends the borrow, so this is fine:
let mut owned = String::from("hello");
let first_two = owned[0..2].to_string();
owned.push('!');
println!("{first_two} / {owned}");


## 4. Literals: Escapes, Raw Strings, Multi-Line

String literals support the escapes you expect — `\n`, `\t`, `\r`, `\\`, `\"`, `\'`, `\0` — plus two Rust-specific forms:

- `\u{1F44B}` — a Unicode scalar value by hex code point.
- `\x41` — a **byte** escape, legal only for values up to `0x7F`, because anything larger would be invalid UTF-8 in a `&str`.

Note there is no `\a`, no octal escape, and no `\e`. If you are porting C, those will not compile.


In [ ]:
// 4.1 The escape inventory, printed in Debug form so the escapes are visible.
let escapes = "tab\there\nnewline\\backslash\"quote";
println!("{escapes:?}");
println!("unicode: {} / {}", "\u{1F44B}", "\u{00E9}");
println!("byte escape: {:?}", "\x41");


### Raw strings: no escapes at all

`r"..."` disables escape processing, which is the right tool for Windows paths, regexes and JSON fragments. When the content itself contains a quote, delimit with hashes: `r#"..."#`, and add more `#` if needed.

The trade is that a raw string cannot contain the sequence that terminates it, and cannot contain a lone `\` at the end.


In [ ]:
// 4.2 Raw strings skip escaping entirely; hashes let the content contain quotes.
println!("{}", r"C:\path\new");
println!("{}", r#"say "hi""#);
println!("{}", r##"a "#" b"##);


### Multi-line strings and line continuation

A newline inside a literal is part of the string. To write a long literal across source lines *without* embedding the newline, end the line with a backslash:

```rust
let long = "first \
            second";      // "first second"
```

Indentation after a continuation is stripped along with the newline and the leading whitespace of the next line. That behaviour is worth knowing precisely, because it is the one place where a string literal's value depends on the source layout.


In [ ]:
// 4.3 Multi-line literals keep newlines; a trailing backslash removes them.
let multi = "line one\nline two";
println!("lines = {}", multi.lines().count());
let joined = "first \
              second";
println!("continuation = {:?}", joined);


### Byte-string literals

`b"abc"` is a `&[u8; 3]` — a byte array, not text. It is the right type for protocol headers, binary magic numbers and byte-level comparisons. The escape rules are the same, but the type carries no UTF-8 guarantee.

To compare a `&str` against bytes, use `.as_bytes()`.


In [ ]:
// 4.4 Byte literals are &[u8; N] — text without the UTF-8 guarantee.
let magic: &[u8; 3] = b"abc";
println!("b\"abc\" = {:?}, len {}", magic, magic.len());
println!("text == bytes: {}", "abc".as_bytes() == magic);


## 5. Growing and Shrinking a `String`

A `String` is a `Vec<u8>`, so it has `len` and `capacity` and grows by reallocation when it runs out. Growing is amortised O(1) per byte, but each growth is a copy — which is why `with_capacity` exists.

| Method | Takes | Effect |
|---|---|---|
| `push(c)` | `char` | Appends one character (1–4 bytes) |
| `push_str(s)` | `&str` | Appends a string slice |
| `insert(i, c)` | index, `char` | Inserts at a byte index; panics if not a boundary |
| `insert_str(i, s)` | index, `&str` | Inserts a slice at a byte index |
| `+= &str` | `&str` | Same as `push_str` |
| `truncate(n)` | `usize` | Shortens to a byte length; panics if not a boundary |
| `clear()` | — | Sets `len` to 0, keeps capacity |
| `pop()` | — | Removes the last `char`, returns `Option<char>` |

`push` and `push_str` are different methods for a reason, and mixing them up produces a distinctive diagnostic — see §13.


In [ ]:
// 5.1 The growth methods: push (char) and push_str (&str) are distinct.
let mut text = String::from("Hello");
text.push_str(", world");
text.push('!');
println!("{text} — len {} cap {}", text.len(), text.capacity());


### Capacity, and why it is visible

The default `String::from("hello")` allocates exactly what it needs: capacity 5. The first `push` then forces a reallocation, typically doubling. That doubling policy is what makes repeated `push` amortised O(1) — but a loop that builds a large string with no pre-reservation still performs O(log n) copies.

`with_capacity(n)` reserves once. `reserve(n)` reserves additional space. `shrink_to_fit()` releases excess — useful when a buffer is retained long-term after being truncated.


In [ ]:
// 5.2 Capacity is observable, and grows by doubling when it is exhausted.
let mut growing = String::new();
let mut previous = growing.capacity();
let mut steps = Vec::new();
for _ in 0..100 {
    growing.push('x');
    if growing.capacity() != previous {
        previous = growing.capacity();
        steps.push(previous);
    }
}
println!("capacity steps while pushing 100 bytes: {steps:?}");
println!("final len {} cap {}", growing.len(), growing.capacity());


### Reserving, clearing, shrinking

Three operations that exist purely to control allocation:

- `String::with_capacity(n)` / `.reserve(n)` — allocate up front.
- `.clear()` — length to zero, **capacity kept**. Reuse the buffer.
- `.shrink_to_fit()` — release unused capacity, possibly reallocating.

`clear()` keeping capacity is the one to internalise for a server: a reused buffer that is cleared each request avoids an allocation per request. The classic pattern is one `String` per connection, cleared between messages.


In [ ]:
// 5.3 reserve allocates once; clear keeps the buffer; shrink gives it back.
let mut reserved = String::new();
reserved.reserve(64);
println!("reserve(64) -> cap {}", reserved.capacity());
let mut reused = String::from("abcdefghij");
reused.clear();
println!("after clear: len {} cap {} (buffer kept)", reused.len(), reused.capacity());
let mut trimmed = String::from("abcdefghij");
trimmed.truncate(2);
trimmed.shrink_to_fit();
println!("after truncate+shrink: len {} cap {}", trimmed.len(), trimmed.capacity());


### Inserting and removing in the middle

`insert`, `insert_str` and `remove` operate on **byte indices** and panic if the index is not a char boundary. They are O(n) because everything after the index shifts. For building text incrementally, appending is almost always the right choice; mid-string insertion is for small edits.

`remove` returns the `char` it removed, which is a nice detail — you get the value back rather than having to read it first.


In [ ]:
// 5.4 Mid-string edits are byte-indexed and O(n).
let mut text = String::from("abc");
text.insert(1, 'X');
text.insert_str(1, "YZ");
println!("after inserts: {text}");
let removed = text.remove(1);
println!("removed {removed:?}, left {text}");


### Mutation through a mutable slice

`get_mut(range)` returns `Option<&mut str>`, which is the safe way to edit a region in place. `str` itself offers `make_ascii_uppercase` and `make_ascii_lowercase`, which modify bytes without reallocating.

These are ASCII-only by name and by behaviour — a non-ASCII character passes through unchanged. That is deliberate: a case mapping that changes byte length cannot be done in place.


In [ ]:
// 5.5 get_mut + make_ascii_uppercase edits in place, without reallocating.
let mut text = String::from("abc");
if let Some(region) = text.get_mut(0..1) {
    region.make_ascii_uppercase();
}
println!("{text} (capacity still {})", text.capacity());
let mut german = String::from("Grüße");
german.make_ascii_uppercase();
println!("make_ascii_uppercase: {german}");


## 6. Iterating: `bytes`, `chars`, `char_indices`

Three iterators cover every iteration need, and picking the wrong one is the most common source of subtle bugs.

| Iterator | Yields | Use for |
|---|---|---|
| `.bytes()` | `u8` | Protocol parsing, ASCII scanning, checksums |
| `.chars()` | `char` | Text semantics, case, predicates |
| `.char_indices()` | `(usize, char)` | Building byte offsets for slicing |

The rule of thumb: **iterate `bytes` when the data is ASCII or binary, `chars` when it is human text, and `char_indices` when you need positions.**

All three are lazy. `.chars()` does not allocate; `.chars().collect::<Vec<char>>()` does, and multiplies memory by up to 4× for ASCII.


In [ ]:
// 6.1 The three iterators over the same string, side by side.
let text = "aé";
println!("bytes        = {:?}", text.bytes().collect::<Vec<u8>>());
println!("chars        = {:?}", text.chars().collect::<Vec<char>>());
println!("char_indices = {:?}", text.char_indices().collect::<Vec<(usize, char)>>());


### Reversing, filtering and collecting

`chars()` composes with the whole iterator toolkit from chapter 03. Reversing a string is `.chars().rev().collect::<String>()` — note that `.bytes().rev()` would produce invalid UTF-8 and therefore does not implement `FromIterator<u8>` for `String`.

Collecting into `String` from `char` is the inverse of `chars()`, and it is also how you assemble a string from computed characters without repeated `push`.


In [ ]:
// 6.2 char pipelines: reverse, filter, and build a String back up.
let reversed: String = "abc".chars().rev().collect();
println!("reversed = {reversed}");
let digits: String = "a1b2c3".chars().filter(|c| c.is_ascii_digit()).collect();
println!("digits = {digits}");
let assembled: String = ['r', 'u', 's', 't'].iter().collect();
println!("assembled = {assembled}");


### Finding a position

`find` returns the **byte index** of the first match, not a character index. This is consistent with slicing — the result is directly usable as an offset — but it surprises anyone coming from Python, where `str.find` returns a character index.

`rfind` searches from the right; `chars().position` gives a character index if you need one.


In [ ]:
// 6.3 find returns a byte offset; position returns a character offset.
let text = "banana";
println!("find(\"an\")  = {:?} (byte index)", text.find("an"));
println!("rfind(\"an\") = {:?} (byte index)", text.rfind("an"));
println!("chars().position(|c| c == 'n') = {:?} (char index)", text.chars().position(|c| c == 'n'));


## 7. The Method Inventory

This section is the reference. Each table groups methods by what they are for; the code cells demonstrate the ones whose behaviour is not obvious from the name.

### Searching and testing

| Method | Returns | Notes |
|---|---|---|
| `contains(p)` | `bool` | `p` may be `&str`, `char` or a predicate |
| `starts_with(p)` / `ends_with(p)` | `bool` | Same pattern types |
| `find(p)` / `rfind(p)` | `Option<usize>` | **Byte** index |
| `is_empty()` | `bool` | `len() == 0` |
| `eq_ignore_ascii_case(other)` | `bool` | ASCII-only, no allocation |
| `is_ascii()` | `bool` | Fast scan |

`contains`, `starts_with` and `find` accept three kinds of pattern: a `char`, a `&str`, or a `&[char]` (any of these characters). This is a trait-based pattern system, not overloading — the same design as the `Pattern` trait.


In [ ]:
// 7.1 Search and test methods, including the three pattern kinds.
let text = "hello world";
println!("contains(\"wor\") = {}", text.contains("wor"));
println!("contains('w')    = {}", text.contains('w'));
println!("contains any of [\"xyz\"] = {}", text.contains(&['x', 'y', 'z'][..]));
println!("starts_with / ends_with = {} / {}", text.starts_with("he"), text.ends_with("ld"));
println!("eq_ignore_ascii_case = {}", "Hello".eq_ignore_ascii_case("hello"));


### Trimming

`trim`, `trim_start` and `trim_end` remove Unicode whitespace. The `_matches` variants remove a **set of characters** instead — `trim_matches('x')` strips leading and trailing `x`, not the substring `"x"`.

That distinction bites: `"xxaxx".trim_matches('x')` is `"a"`, while `"xxaxx".trim_matches("xx")` also strips repeated *substrings*. Use the `char` form unless you specifically need the substring behaviour.


In [ ]:
// 7.2 trim removes whitespace; trim_matches removes a character set.
println!("trim        = {:?}", "  hi  ".trim());
println!("trim_start  = {:?}", "  hi  ".trim_start());
println!("trim_matches('x') = {:?}", "xxhixx".trim_matches('x'));
println!("trim_start_matches('0') = {:?}", "0012".trim_start_matches('0'));
println!("trim_end_matches('0')   = {:?}", "1200".trim_end_matches('0'));


### Case conversion, and the Unicode trap

There are two families, and confusing them is a correctness bug rather than a style issue:

| Method | Domain | Can change length | Example |
|---|---|---|---|
| `to_ascii_uppercase()` | ASCII only | No | `"ß"` → `"ß"` (unchanged) |
| `to_uppercase()` | Full Unicode | **Yes** | `"ß"` → `"SS"` |
| `to_ascii_lowercase()` | ASCII only | No | `"ABC"` → `"abc"` |
| `to_lowercase()` | Full Unicode | **Yes** | `"İ"` → `"i\u{307}"` |

The Unicode forms allocate and can lengthen the string. The ASCII forms are in-place-capable and byte-count preserving. For protocol tokens, command names and HTTP headers — which are ASCII by specification — the ASCII forms are both correct and cheaper.

Note also that `to_uppercase()` maps a single `char` to a `String`, because one character can become several. `char::to_uppercase()` returns an iterator for the same reason.


In [ ]:
// 7.3 ASCII case conversion is length-preserving; Unicode case conversion is not.
println!("to_uppercase(\"ß\")        = {:?}", "ß".to_uppercase());
println!("to_ascii_uppercase(\"ß\")  = {:?}", "ß".to_ascii_uppercase());
println!("to_uppercase(\"Grüße\")    = {:?}", "Grüße".to_uppercase());
println!("to_ascii_uppercase(...)  = {:?}", "Grüße".to_ascii_uppercase());
println!("'ß'.to_uppercase() collects to {:?}", 'ß'.to_uppercase().collect::<String>());


### Prefix, suffix and splitting on a separator

| Method | Returns | Notes |
|---|---|---|
| `strip_prefix(p)` | `Option<&str>` | `None` if absent — no panic |
| `strip_suffix(p)` | `Option<&str>` | As above |
| `split_once(p)` | `Option<(&str, &str)>` | Splits at the **first** occurrence |
| `rsplit_once(p)` | `Option<(&str, &str)>` | Splits at the **last** |

`strip_prefix` and `strip_once` are the parsing primitives you want: they return `Option`, they borrow rather than allocate, and they compose with `?` — which is why §11 can write a key/value parser in four lines.


In [ ]:
// 7.4 strip_* and split_once return Options and borrow rather than allocate.
println!("strip_prefix = {:?}", "foo.rs".strip_prefix("foo"));
println!("strip_suffix = {:?}", "foo.rs".strip_suffix(".rs"));
println!("strip miss   = {:?}", "foo.rs".strip_prefix("bar"));
println!("split_once   = {:?}", "key=value=1".split_once('='));
println!("rsplit_once  = {:?}", "key=value=1".rsplit_once('='));


### Replacing

| Method | Replaces | Notes |
|---|---|---|
| `replace(from, to)` | All occurrences | Returns a new `String` |
| `replacen(from, to, n)` | First *n* | Returns a new `String` |

Both allocate, because the result length is unknown up front. `from` and `to` may each be a `char` or a `&str`, independently — `replace('a', "bb")` is legal and changes the length.

For a hot path, `str::matches(p).count()` first lets you decide whether to allocate at all.


In [ ]:
// 7.5 replace and replacen always allocate and return a new String.
println!("replace(char, &str) = {:?}", "aaa".replace('a', "b"));
println!("replace(&str, &str) = {:?}", "abcabc".replace("ab", "X"));
println!("replacen(..., 2)    = {:?}", "aaa".replacen('a', "b", 2));


### Allocation and buffer control

| Method | Purpose |
|---|---|
| `with_capacity(n)` | Allocate up front |
| `reserve(n)` | Ensure room for `n` more bytes |
| `capacity()` | Report the allocation |
| `shrink_to_fit()` | Release excess |
| `clear()` | Length 0, capacity kept |
| `truncate(n)` | Cut to a byte length |

The pattern for a long-lived buffer in a server is `clear()` between uses and `reserve` once at construction. `shrink_to_fit` is for when the peak was large and the steady state is small.


In [ ]:
// 7.6 A reusable buffer: reserve once, clear between messages, never reallocate.
let mut buffer = String::with_capacity(64);
for message in ["first", "second message"] {
    buffer.clear();
    buffer.push_str(message);
    println!("{buffer:?} (cap {})", buffer.capacity());
}


### Character predicates

`char` has a large predicate surface, and the ASCII-suffixed variants are both faster and more predictable than the Unicode ones:

| Unicode | ASCII-only | True for |
|---|---|---|
| `is_alphabetic()` | `is_ascii_alphabetic()` | Letters |
| `is_numeric()` | `is_ascii_digit()` | Digits |
| `is_alphanumeric()` | `is_ascii_alphanumeric()` | Letters or digits |
| `is_whitespace()` | `is_ascii_whitespace()` | Whitespace |
| — | `is_ascii_hexdigit()` | `0-9a-fA-F` |
| — | `is_ascii_punctuation()` | ASCII punctuation |

`is_numeric()` is true for Unicode numeric characters such as `'³'`, which `is_ascii_digit()` rejects. When validating a wire format, the ASCII predicate is what you want.


In [ ]:
// 7.7 ASCII predicates are narrower and faster than their Unicode counterparts.
println!("'a' alphabetic = {}, '7' ascii_digit = {}", 'a'.is_alphabetic(), '7'.is_ascii_digit());
println!("'³' is_numeric = {}, '³' is_ascii_digit = {}", '³'.is_numeric(), '³'.is_ascii_digit());
println!("'f'.to_digit(16) = {:?}, 'g'.to_digit(16) = {:?}", 'f'.to_digit(16), 'g'.to_digit(16));
println!("' '.is_whitespace = {}, '\\u{{a0}}'.is_whitespace = {}", ' '.is_whitespace(), '\u{a0}'.is_whitespace());


### `char` ↔ number conversions

`char` is a Unicode scalar value, so it converts to and from `u32` — but not implicitly. The conversions are explicit and checked:

| Conversion | Result | Notes |
|---|---|---|
| `'a' as u32` | `97` | Always valid |
| `65u8 as char` | `'A'` | `as` truncates and may produce a replacement char |
| `char::from_u32(n)` | `Option<char>` | `None` for surrogates and out-of-range |
| `'f'.to_digit(16)` | `Option<u32>` | `None` if not a digit in that base |

`char::from_u32` is the correct conversion for untrusted input; `as char` is the fast one that can silently produce `'\u{FFFD}'` for a value that is not a valid scalar.


In [ ]:
// 7.8 char/number conversions: `as` is unchecked, from_u32 is checked.
println!("'a' as u32 = {}, 65u8 as char = {}", 'a' as u32, 65u8 as char);
println!("from_u32(0x1F44B) = {:?}", char::from_u32(0x1F44B));
println!("from_u32(0xD800)  = {:?} (surrogate)", char::from_u32(0xD800));
println!("to_digit(2) of '1' = {:?}", '1'.to_digit(2));


## 8. Splitting, Lines and Joining

`split` takes a pattern and yields subslices — **borrowed** views into the original, with no allocation. That is the property to internalise: splitting a `&str` produces `&str`s, and the original must outlive them.

| Method | Yields | Notes |
|---|---|---|
| `split(p)` | `&str` | Keeps empty fields |
| `split_whitespace()` | `&str` | Collapses runs, skips empties |
| `splitn(n, p)` | `&str` | At most `n` pieces |
| `rsplitn(n, p)` | `&str` | From the right |
| `split_inclusive(p)` | `&str` | Keeps the separator |
| `lines()` | `&str` | Splits on `\n`, strips a trailing `\r` |
| `split_terminator(p)` | `&str` | Like `split`, drops a trailing empty |

`split(',')` on `"a,b,,c"` yields four items including the empty one. `split_whitespace()` on `"  a  b\tc "` yields three. Choose deliberately: one is a format parser, the other is a tokenizer.


In [ ]:
// 8.1 split keeps empty fields; split_whitespace collapses runs.
println!("split(',')        = {:?}", "a,b,,c".split(',').collect::<Vec<&str>>());
println!("split_whitespace  = {:?}", "  a  b\tc ".split_whitespace().collect::<Vec<&str>>());
println!("splitn(2, ':')    = {:?}", "a:b:c".splitn(2, ':').collect::<Vec<&str>>());
println!("rsplitn(2, ':')   = {:?}", "a:b:c".rsplitn(2, ':').collect::<Vec<&str>>());
println!("split_inclusive   = {:?}", "a,b,".split_inclusive(',').collect::<Vec<&str>>());


### Lines, and the `\r\n` question

`lines()` splits on `\n` and removes a trailing `\r`. That single behaviour is what makes it correct for both Unix and Windows text files without special-casing — and it is why `lines()` is preferred over `split('\n')` for reading files.

The counter-example is a file with only `\r` line endings (classic Mac), which `lines()` does not split. That format is extinct in practice, but worth knowing the boundary of the guarantee.


In [ ]:
// 8.2 lines() strips a trailing \r, so it handles CRLF and LF alike.
println!("lines()      = {:?}", "a\r\nb\nc".lines().collect::<Vec<&str>>());
println!("split('\\n')  = {:?}", "a\r\nb\nc".split('\n').collect::<Vec<&str>>());


### Joining, and building text from pieces

`join` is on slices of strings, not on `String`: `["a", "b"].join("-")`. It allocates exactly once if you use `.join()`, which makes it the right tool for assembling output from parts — far better than a loop of `push_str` with manual separator handling.

`concat` is the separator-free version, also on slices.


In [ ]:
// 8.3 join and concat build from slices in one allocation.
println!("join   = {}", ["a", "b", "c"].join("-"));
println!("concat = {}", ["a", "b", "c"].concat());
println!("repeat = {}", "ab".repeat(3));


## 9. Converting Between Bytes and Text

This is where UTF-8 validation becomes explicit. Every conversion from bytes to text can fail, and the standard library gives you three levels of strictness:

| Function | Input | On invalid UTF-8 |
|---|---|---|
| `std::str::from_utf8(&[u8])` | borrowed bytes | `Result<&str, Utf8Error>` |
| `String::from_utf8(Vec<u8>)` | owned bytes | `Result<String, FromUtf8Error>` |
| `String::from_utf8_lossy(&[u8])` | borrowed bytes | Replaces bad bytes with `U+FFFD` |

`from_utf8_lossy` returns a `Cow<str>` — borrowed if the input was already valid, owned if replacement was needed. That is the "avoid the copy when possible" type, and it is why it can be used on a hot path without always allocating.


In [ ]:
// 9.1 The three levels: strict borrowed, strict owned, and lossy.
println!("from_utf8 valid   = {:?}", std::str::from_utf8(&[104u8, 105]));
println!("from_utf8 invalid = {:?}", std::str::from_utf8(&vec![0xffu8]));
println!("String::from_utf8 = {:?}", String::from_utf8(vec![104u8, 105]));
println!("lossy             = {:?}", String::from_utf8_lossy(&vec![0x66u8, 0xff, 0x6fu8]));


### Reading the error

`Utf8Error` tells you exactly where validation failed, which is what you need to report a line and column:

| Field / method | Meaning |
|---|---|
| `valid_up_to()` | Byte index of the first invalid sequence |
| `error_len()` | `Some(n)` for an invalid sequence, `None` for truncated input |

`error_len() == None` means the input ended in the middle of a multi-byte sequence — a truncated read, not a corrupt byte. Distinguishing those two is the difference between "retry with more data" and "reject the message".


In [ ]:
// 9.2 Utf8Error reports where validation stopped, and whether input was truncated.
// The bytes are built at run time so that this is a real check, not a constant one.
let mut cut = "h\u{e9}".as_bytes().to_vec();
cut.pop();                                  // truncates mid-sequence
let truncated = std::str::from_utf8(&cut).unwrap_err();
println!("truncated: valid_up_to {}, error_len {:?}", truncated.valid_up_to(), truncated.error_len());
let bad = vec![0x68u8, 0xff];               // a corrupt byte, not a short read
let corrupt = std::str::from_utf8(&bad).unwrap_err();
println!("corrupt:   valid_up_to {}, error_len {:?}", corrupt.valid_up_to(), corrupt.error_len());


### The zero-copy round trip

`as_bytes()` is free — it reinterprets the same buffer. `into_bytes()` consumes a `String` and hands back its `Vec<u8>`, also free. Together they let you move between text and bytes without copying:

```text
String  --into_bytes()-->  Vec<u8>   (moves the allocation)
&str    --as_bytes()--->   &[u8]     (borrows the allocation)
```

The reverse direction is the one that validates. That asymmetry — free out, checked back in — is the design: producing UTF-8 from known-good text cannot fail, while accepting bytes from the outside world must be verified.


In [ ]:
// 9.3 Bytes out is free; text in is validated.
let owned = String::from("héllo");
let bytes = owned.clone().into_bytes();
println!("into_bytes = {:?}", bytes);
println!("as_bytes   = {:?}", "é".as_bytes());
let back = String::from_utf8(bytes).unwrap();
println!("round trip = {back}");


## 10. Passing Strings: `&str` in, `String` out

The single most useful API convention in Rust is this:

> **Accept `&str`, return `String`.**

Accepting `&str` means the caller can pass a literal, a `&String` (via deref coercion) or a slice, without any conversion. Returning `String` means the function owns its result and the caller is not tied to the function's internals.

| Signature | Accepts | Verdict |
|---|---|---|
| `fn f(s: &str)` | `&str`, `&String`, literal, slice | Preferred |
| `fn f(s: String)` | Only an owned `String` | Forces the caller to allocate or move |
| `fn f(s: &String)` | `&String` only | Almost always a mistake |

`&String` is the one to avoid: it is strictly less useful than `&str` and buys nothing, because deref coercion goes `&String → &str` but not the reverse.


In [ ]:
// 10.1 Taking &str accepts every string-like value without conversion.
fn byte_len(text: &str) -> usize {
    text.len()
}
let owned = String::from("deref");
println!("literal      = {}", byte_len("literal"));
println!("&String      = {}", byte_len(&owned));
println!("String::as_str = {}", byte_len(owned.as_str()));
println!("slice        = {}", byte_len(&owned[0..2]));


### Why `&str` parameters are not just convenient

A `&str` parameter makes the function's **ownership intent** visible: it borrows, so it cannot consume or free the caller's data. A `String` parameter silently transfers ownership, which is sometimes right (a builder that stores the value) and usually unnecessary.

The same reasoning applies to return values. Returning `&str` from a function that creates text is impossible without a lifetime relationship to something that outlives the call — which is why constructors return `String`:

```text
error[E0106]: missing lifetime specifier
   |
   | fn dangling() -> &str {
   |                  ^ expected named lifetime parameter
   |
   = help: this function's return type contains a borrowed value,
           but there is no value for it to be borrowed from
   = help: instead, you are more likely to want to return an owned value
```

That last hint is the whole lesson: if you build it, you own it, so return `String`.


In [ ]:
// 10.2 Building text means returning String; borrowing text means taking &str.
fn shout(text: &str) -> String {
    let mut owned = text.to_owned();
    owned.push('!');
    owned
}
fn first_word(text: &str) -> &str {
    match text.find(' ') {
        Some(index) => &text[..index],
        None => text,
    }
}
println!("shout = {:?}", shout("hello"));
println!("first_word = {:?}", first_word("hello world"));


## 11. Text Processing Patterns

The methods from §7 and §8 compose into the parsing idioms you will use constantly. This section is the "what does this actually look like in code" companion to the reference tables.

### Pattern 1: split, trim, parse

Parsing a delimited line is the most common text task in systems code. `split_once` plus `?` plus `.parse().ok()?` expresses "this is well-formed or there is no result" without a single explicit match:

```rust
fn parse_kv(line: &str) -> Option<(&str, i64)> {
    let (key, value) = line.split_once('=')?;
    Some((key.trim(), value.trim().parse().ok()?))
}
```

Every `?` here is a short-circuit: a missing separator, or a value that does not parse, yields `None`. That is `Option` propagation, and it is the reason `split_once` returns `Option` rather than panicking.


In [ ]:
// 11.1 Parsing with split_once, trim and ?, with no explicit match.
fn parse_kv(line: &str) -> Option<(&str, i64)> {
    let (key, value) = line.split_once('=')?;
    Some((key.trim(), value.trim().parse().ok()?))
}
println!("well-formed = {:?}", parse_kv("  latency_ns = 250  "));
println!("no separator = {:?}", parse_kv("broken"));
println!("bad value = {:?}", parse_kv("k=x"));


### Pattern 2: the iterator pipeline

`split_whitespace()` yields borrowed slices, and the chapter 03 iterator combinators consume them without allocating intermediate collections. `.map(|w| w.len()).sum()` walks once and allocates nothing.

Compare with the Python idiom, which builds a list of strings and then a list of lengths. Rust's iterators are lazy, so the intermediate `Vec` never exists unless you ask for it with `.collect()`.


In [ ]:
// 11.2 An iterator pipeline over borrowed words: no intermediate collection.
let text = "the quick brown fox jumps over the lazy dog";
let words: Vec<&str> = text.split_whitespace().collect();
let longest = words.iter().copied().max_by_key(|word| word.len()).unwrap_or("");
let total: usize = text.split_whitespace().map(|word| word.len()).sum();
println!("words = {}, longest = {longest:?}, total bytes = {total}", words.len());


### Pattern 3: normalising keys

Configuration keys, header names and command tokens are conventionally case-insensitive and ASCII. The normalisation is `trim().to_ascii_lowercase()`, and it is worth noting what it does *not* do: no Unicode case folding, no allocation for the trim (the lowercase does allocate).

For a comparison-only use case, skip the allocation entirely with `eq_ignore_ascii_case`.

| Goal | Idiom | Allocates |
|---|---|---|
| Store a normalised key | `key.trim().to_ascii_lowercase()` | Yes |
| Compare case-insensitively | `a.eq_ignore_ascii_case(b)` | No |


In [ ]:
// 11.3 Normalise for storage, or compare in place without allocating.
let header = "  Content-Type  ";
println!("normalised key = {:?}", header.trim().to_ascii_lowercase());
println!("compare in place = {}", "Content-Type".eq_ignore_ascii_case("content-type"));


### Pattern 4: counting and grouping

`split` plus `fold` builds a `HashMap` in one pass. This is the shape of a log summariser, a message-type histogram, or a fill counter — the kind of thing you write constantly against market data.

The important detail is that the keys are `&str` borrowed from the input, so no string is copied for the key itself.


In [ ]:
// 11.4 One-pass grouping into a HashMap with borrowed keys.
let events = "trade,quote,trade,cancel,quote,trade";
let mut counts: std::collections::HashMap<&str, u32> = std::collections::HashMap::new();
for event in events.split(',') {
    *counts.entry(event).or_insert(0) += 1;
}
let mut summary: Vec<(&str, u32)> = counts.into_iter().collect();
summary.sort();
println!("{summary:?}");


## 12. Formatting and Display

`format!` builds a `String`; `println!` writes to stdout; `write!` writes into any `fmt::Write` — including a `String`. All three share the same format-specification language, so anything you learn for one applies to the others.

| Macro | Destination | Returns |
|---|---|---|
| `format!` | New `String` | `String` |
| `println!` / `print!` | stdout | `()` |
| `eprintln!` / `eprint!` | stderr | `()` |
| `write!(buf, ...)` | Any `fmt::Write` | `Result` |
| `dbg!(x)` | stderr | The value, unchanged |

`write!` into a `String` is the zero-surprise way to build formatted output incrementally, and it is infallible for `String` — the `Result` exists because `fmt::Write` also covers I/O-backed writers.


In [ ]:
// 12.1 write! appends into a String with the same format language as println!.
let mut row = String::new();
std::fmt::Write::write_fmt(&mut row, format_args!("{:>5}|{:<5}|{:^5}|", 42, "ab", "mid")).unwrap();
println!("{row:?}");


### The format spec, in one table

```text
{ [argument] [:[fill][align][sign][#][0][width][.precision][type]] }
```

| Spec | Meaning | Example output |
|---|---|---|
| `{:>8}` | Right-align in 8 columns | `"   right"` |
| `{:<8}` | Left-align | `"left    "` |
| `{:^8}` | Centre | `"  mid   "` |
| `{:08}` | Zero-pad to width 8 | `"00000042"` |
| `{:+}` | Always show sign | `"+42"` |
| `{:#x}` / `{:#b}` | Hex / binary with prefix | `"0xff"`, `"0b101"` |
| `{:.3}` | Precision (floats: decimals; strings: max chars) | `"3.142"` |
| `{:?}` | `Debug` | `"\"a\\tb\""` |
| `{:#?}` | Pretty `Debug` | Multi-line |

Note that `.precision` on a **string** truncates to at most that many *characters* — not bytes — which makes it safe on Unicode. That is a rare case of the formatter being Unicode-aware.


In [ ]:
// 12.2 Alignment, padding, sign, radix and precision in one place.
println!("[{:>8}][{:<8}][{:^8}]", "right", "left", "mid");
println!("[{:08}][{:+}][{:#x}][{:#b}]", 42, 42, 255, 5);
println!("[{:.3}]", 3.14159);
println!("string precision truncates chars: [{:.3}]", "truncated");


### `Display` versus `Debug` for strings

For a `&str`, the two are visibly different, and this is the fastest way to tell them apart:

| Format | `"a\tb"` prints as | Purpose |
|---|---|---|
| `{}` (`Display`) | `a	b` | Human-facing text |
| `{:?}` (`Debug`) | `"a\tb"` | Escaped, unambiguous |

`Debug` is what you want in logs and in assertion messages, because it shows the escapes and the delimiters. `Display` is what you want in output. `{:#?}` extends `Debug` to multiple lines and is the right choice for nested structures.


In [ ]:
// 12.3 Display shows text; Debug shows escapes and quotes.
let text = "a\tb";
println!("Display: [{text}]");
println!("Debug:   [{text:?}]");
println!("escapes: {}", "é".escape_debug());


### Building output without a loop of `push_str`

Two idioms are worth preferring over manual concatenation:

- `write!(buffer, ...)` — formatted append, one call per piece.
- `slice.join(sep)` — separator handling for free, one allocation.

Both avoid the class of bug where a separator is emitted before the first element or after the last.


In [ ]:
// 12.4 Two idioms that beat manual concatenation.
let fields = ["BTC-USD", "buy", "0.5"];
let line = fields.join(";");
println!("join: {line}");
let mut report = String::new();
std::fmt::Write::write_fmt(&mut report, format_args!("orders={} symbols={}", fields.len(), 1)).unwrap();
println!("write!: {report}");


## 13. Reading the Compiler on Strings

Four diagnostics account for most of the string errors you will hit. Each one is short, and each names its own fix. Reading them is faster than searching for them.

| Code | Trigger | Fix |
|---|---|---|
| `E0277` | `s[0]` | Range, `.chars().nth(0)`, or `.as_bytes()[0]` |
| `E0308` | `String + String` | `a + &b`, or `format!` |
| `E0308` | `push_str('c')` | `push('c')` or `push_str("c")` |
| `E0502` | Mutating a borrowed `String` | End the borrow, or copy out |
| `E0382` | Using `a` after `a + &b` moved it | Reorder, or `clone()` |
| `E0505` | Moving while borrowed | Drop the borrow first |
| `E0106` | Returning `&str` built from a local | Return `String` |

The rest of this section shows the four most common ones verbatim.


### `E0277` — the type `str` cannot be indexed by `{integer}`

```text
error[E0277]: the type `str` cannot be indexed by `{integer}`
   = note: you can use `.chars().nth()` or `.bytes().nth()`
```

Cause: `s[0]`. Fix: a range (`&s[0..1]`), `.chars().nth(0)`, or `.as_bytes()[0]` if you really mean a byte.


### `E0308` — expected `&str`, found `String`

```text
error[E0308]: mismatched types
   |
   |     let c = a + b;
   |                  ^ expected `&str`, found `String`
   |
help: consider borrowing here
   |
   |     let c = a + &b;
```

Cause: `String + String`. `Add` for `String` is implemented for `&str`, not `String`. Fix: `a + &b`, or `format!`.


### `E0308` — expected `&str`, found `char`

```text
error[E0308]: mismatched types
   |
   |     s.push_str('b');
   |                ^^^ expected `&str`, found `char`
   |
help: if you meant to write a string literal, use double quotes
```

Cause: `push_str` takes a slice, `push` takes a `char`. Fix: `s.push('b')` or `s.push_str("b")`.


### `E0502` — mutable borrow while immutably borrowed

```text
error[E0502]: cannot borrow `s` as mutable because it is also borrowed as immutable
   |
   |     let slice = &s[0..2];
   |                  - immutable borrow occurs here
   |     s.push('!');
   |     ^^^^^^^^^^^ mutable borrow occurs here
```

Cause: a live slice, then a mutation. Fix: copy the slice out, or finish with it first. Chapter 05 explains the rule in full.


In [ ]:
// 13.1 The four diagnostics are comments; the correct spellings run here.
//
//     let c = s[0];                 // E0277: `str` cannot be indexed by `{integer}`
//     let c = a + b;                // E0308: expected `&str`, found `String`
//     s.push_str('b');              // E0308: expected `&str`, found `char`
//     let slice = &s[0..2]; s.push('!');   // E0502: borrow conflict
//
let s = String::from("hello");
println!("byte 0        = {}", s.as_bytes()[0]);
println!("char 0        = {:?}", s.chars().nth(0));
println!("concat        = {}", String::from("a") + &String::from("b"));
println!("push vs push_str = {}", { let mut t = String::from("a"); t.push('b'); t.push_str("c"); t });


### The pattern in all four

Notice what the diagnostics have in common: they do not say "you are wrong", they say **which type was expected and which was found**, and then offer the coercion that would bridge them. Rust's string API is small; almost every error is one of these four, and each has a mechanical fix.

The habit worth forming: read the `expected ... found ...` line first, then the `help:` block. Between them, the fix is usually already written.


## 14. Unicode Beyond ASCII

This section is deliberately short, and its message is a warning rather than a technique.

### Grapheme clusters: the layer Rust does not have

A `char` is a **scalar value**, not a *user-perceived character*. Some visible characters are several scalar values:

| Text | `chars().count()` | `len()` | User sees |
|---|---|---|---|
| `"é"` (precomposed) | 1 | 2 | one character |
| `"e\u{301}"` (e + combining accent) | 2 | 3 | one character |
| `"👋🏽"` (wave + skin tone) | 2 | 8 | one character |
| `"🇫🇷"` (two regional indicators) | 2 | 8 | one flag |

Consequently: **`s.len()` is never the number of characters the user sees**, and even `s.chars().count()` is not. Grapheme cluster segmentation requires Unicode table data that `std` does not ship, which is why the `unicode-segmentation` crate exists. This series uses `std` only, so the honest statement is: Rust gives you bytes and scalar values, and everything above that is a library's job.

This matters for HFT in a specific way: if you are validating or truncating a *field* of a wire protocol, the field is defined in bytes. If you are rendering text to a human, you need grapheme clusters. Do not mix the two.


In [ ]:
// 14.1 The same visible character, in two encodings, with different lengths.
let precomposed = "é";
let decomposed = "e\u{301}";
println!("precomposed: {} bytes, {} chars", precomposed.len(), precomposed.chars().count());
println!("decomposed:  {} bytes, {} chars", decomposed.len(), decomposed.chars().count());
println!("byte-equal = {}", precomposed == decomposed);


### Case mapping can change length

`.to_uppercase()` is a Unicode-aware, *length-changing* operation: it returns an iterator because one `char` can map to several. `'ß'` uppercases to `"SS"`. `to_ascii_uppercase()` cannot change length, and leaves `'ß'` alone.

The rule: for identifiers and protocol tokens (ASCII), use the `ascii_` variants — they are faster and cannot surprise you. For display text, use the Unicode variants.


In [ ]:
// 14.2 Unicode case mapping can lengthen; the ASCII variant cannot.
println!("'ß'.to_uppercase()       = {:?}", 'ß'.to_uppercase().collect::<String>());
println!("'ß'.to_ascii_uppercase() = {:?}", 'ß'.to_ascii_uppercase());
println!("\"Grüße\".to_uppercase()   = {:?}", "Grüße".to_uppercase());
println!("\"Grüße\".to_ascii_uppercase() = {:?}", "Grüße".to_ascii_uppercase());


### `char` validation: surrogates are not scalar values

`char::from_u32` rejects the UTF-16 surrogate range `0xD800..=0xDFFF` and anything above `0x10FFFF`, returning `None`. This is exactly the validation you need when a protocol hands you a 32-bit code point that might be junk: `char::from_u32` is a total function from `u32` to "valid scalar value or `None`", and it never panics.

The inverse for output is `char::encode_utf16`, which fills a `[u16; 2]` and returns the slice that was used — the two-slot return is how a `char` above the BMP encodes.


In [ ]:
// 14.3 Validate code points with char::from_u32; encode to UTF-16.
println!("U+1F44B = {:?}", char::from_u32(0x1F44B));
println!("U+D800 (surrogate) = {:?}", char::from_u32(0xD800));
println!("U+110000 (too high) = {:?}", char::from_u32(0x110000));
let mut buf = [0u16; 2];
let encoded = '👋'.encode_utf16(&mut buf);
println!("encode_utf16 = {encoded:?}");


## 15. Performance Notes for Low-Latency Code

Everything in this section follows from one fact: a `String` is a heap allocation, and `&str` is a borrow of bytes. On a latency-sensitive path, the number of allocations per message is a first-class design parameter.

| Cost | Operation |
|---|---|
| Free | `as_bytes()`, `as_str()`, `len()`, `is_empty()`, comparison |
| No allocation, O(n) walk | `chars()`, `bytes()`, `find`, `starts_with` |
| Allocates | `to_string()`, `to_uppercase()`, `format!`, `replace`, `split().collect()` |
| May reallocate | `push`, `push_str`, `+` |

The three rules that matter most:

1. **Parse bytes, not characters.** If the protocol is ASCII — FIX, CSV, most log lines — work with `as_bytes()` and byte offsets. `find` and `memchr`-style scanning never decode UTF-8, so they never pay for it.
2. **Borrow instead of owning.** A function that takes `&str` accepts `&String` and `&str` alike and allocates nothing. Only return `String` when you genuinely produce new text.
3. **Reuse the buffer.** `clear()` keeps the capacity, so a parsing loop that clears and refills a `String` performs one allocation instead of one per message.


In [ ]:
// 15.1 Reusing one buffer across many messages: allocation count stays flat.
let messages = ["a=1", "bb=22", "ccc=333", "dddd=4444"];
let mut buffer = String::with_capacity(32);
let start_capacity = buffer.capacity();
for message in messages {
    buffer.clear();
    buffer.push_str(message);
    let _ = buffer.split_once('=').map(|(k, v)| (k.len(), v.len()));
}
println!("capacity before = {start_capacity}, after = {}", buffer.capacity());
println!("final buffer = {buffer:?}");


### ASCII predicates and comparisons avoid the Unicode machinery

`to_lowercase()` allocates and consults Unicode tables. When the input is known ASCII, three alternatives are strictly better:

| Instead of | Use | Why |
|---|---|---|
| `a.to_lowercase() == b.to_lowercase()` | `a.eq_ignore_ascii_case(b)` | No allocation, no tables |
| `c.to_ascii_lowercase()` in a loop | `c.eq_ignore_ascii_case(&other)` | No per-char work |
| `s.chars().all(\|c\| c.is_digit(10))` | `s.bytes().all(\|b\| b.is_ascii_digit())` | Byte loop, no decode |

This is not micro-optimisation for its own sake: on a hot path it is the difference between a comparison and an allocation, and allocations are where the tail latencies come from.


In [ ]:
// 15.2 ASCII-level checks: no allocation, no Unicode tables.
let token = "BUY";
println!("eq_ignore_ascii_case = {}", token.eq_ignore_ascii_case("buy"));
println!("all digits (bytes)   = {}", "20250101".bytes().all(|b| b.is_ascii_digit()));
println!("no allocation from   = {:?}", token.as_bytes());


### `Cow<str>`: borrowed when you can, owned when you must

`std::borrow::Cow<'_, str>` is "either a borrow or an owned value". It is the type you reach for when a function *sometimes* needs to produce new text and *usually* does not — normalising a key that is already lowercase, or unescaping a string that contains no escapes.

```rust
fn normalise(raw: &str) -> Cow<'_, str> {
    if raw.bytes().all(|b| !b.is_ascii_uppercase()) {
        Cow::Borrowed(raw)          // no allocation
    } else {
        Cow::Owned(raw.to_ascii_lowercase())   // allocate once
    }
}
```

Both arms have type `Cow<str>`, so the caller writes one code path and pays for an allocation only on the inputs that need it.


In [ ]:
// 15.3 Cow<str> stays borrowed on the fast path and owns only when needed.
use std::borrow::Cow;
fn normalise(raw: &str) -> Cow<'_, str> {
    if raw.bytes().all(|b| !b.is_ascii_uppercase()) {
        Cow::Borrowed(raw)
    } else {
        Cow::Owned(raw.to_ascii_lowercase())
    }
}
let already = normalise("btc-usd");
let changed = normalise("BTC-USD");
println!("already = {already:?} borrowed = {}", matches!(already, Cow::Borrowed(_)));
println!("changed = {changed:?} owned = {}", matches!(changed, Cow::Owned(_)));


### `&str` keys borrow the input; `String` keys copy it

In §11.4 the `HashMap` used `&str` keys pointing into the source string. That is free. The moment you build a map keyed by `String`, every key is a heap allocation, and every lookup of an existing key may allocate too.

| Key type | Lookup allocates | Lifetime tied to |
|---|---|---|
| `&str` | No | The source buffer |
| `String` | Only on insert | Nothing (owned) |

The trade is real: `&str` keys cannot outlive the buffer they point into. For per-message processing, the buffer is reused and the map is cleared — so `&str` keys are both correct and allocation-free. For long-lived caches, `String` is the price of ownership.


### Why `.chars()` walks the whole string

`chars()` is not O(1) to advance: it must decode a variable-length sequence, and it must validate. So `s.chars().nth(1000)` is O(1000) — there is no character index, only a byte index. `s.chars().count()` is O(n) and is *not* cached.

If you need random access by character position, `Vec<char>` buys it at the cost of 4 bytes per character plus a copy:

| Access pattern | Use | Cost |
|---|---|---|
| Sequential | `chars()` | O(n), no allocation |
| Random by char index | `chars().nth(i)` | O(i) |
| Random, repeated | `Vec<char>` | O(n) build, O(1) access |

For HFT, the answer is usually none of these: you want byte offsets, because the protocol is defined in bytes.


In [ ]:
// 15.4 Byte offsets are O(1); char offsets are O(n) walks.
let line = "symbol=ES,price=5000.25";
let at = line.find("price").unwrap();
println!("byte offset of price = {at}");
println!("char at that offset = {:?}", line.chars().nth(at));
println!("as_bytes()[{at}..] = {:?}", std::str::from_utf8(&line.as_bytes()[at..at + 5]).unwrap());


## 16. Cheat Sheet

### The two types

| Type | Owns | Mutable | Where it lives |
|---|---|---|---|
| `String` | Yes (heap `Vec<u8>`) | Yes, if `let mut` | Everywhere |
| `&str` | No (borrowed bytes) | No | Everywhere |
| `&mut str` | No | In-place ASCII only | Rare |

A string literal `"abc"` is a `&'static str`. `&String` derefs to `&str` automatically in function arguments.

### The three iterators

| Call | Yields | Index unit | Cost |
|---|---|---|---|
| `.bytes()` | `u8` | Byte | O(n) |
| `.chars()` | `char` | Scalar value | O(n), decodes |
| `.char_indices()` | `(usize, char)` | Byte offset of the char | O(n), decodes |

### Which length

| Question | Answer |
|---|---|
| Bytes? | `s.len()` |
| Scalar values? | `s.chars().count()` |
| Grapheme clusters? | Not in `std` |

### Construction

| Need | Write |
|---|---|
| Empty, grows | `String::new()` |
| Empty, known size | `String::with_capacity(n)` |
| From literal | `String::from("x")` / `"x".to_string()` |
| From owned bytes | `String::from_utf8(v)?` |
| From bytes, lossy | `String::from_utf8_lossy(&v)` |

### Growing and shrinking

| Need | Write |
|---|---|
| Append `char` | `s.push('c')` |
| Append `&str` | `s.push_str("x")` |
| Insert at byte offset | `s.insert(i, 'c')` / `s.insert_str(i, "x")` |
| Remove `char` | `s.remove(i)` → `char` |
| Truncate to byte offset | `s.truncate(i)` |
| Empty but keep capacity | `s.clear()` |
| Release unused capacity | `s.shrink_to_fit()` |

### Searching and testing

| Need | Write |
|---|---|
| Substring byte offset | `s.find(p)` / `s.rfind(p)` → `Option<usize>` |
| Contains | `s.contains(p)` |
| Prefix / suffix | `s.starts_with(p)` / `s.ends_with(p)` |
| Strip prefix / suffix | `s.strip_prefix(p)` / `s.strip_suffix(p)` → `Option<&str>` |
| Split at first / last | `s.split_once(p)` / `s.rsplit_once(p)` → `Option<(&str, &str)>` |
| Case-insensitive ASCII equality | `a.eq_ignore_ascii_case(b)` |

`p` may be a `char`, a `&str`, or a predicate `|c: char| -> bool`.

### Trimming and replacing

| Need | Write |
|---|---|
| Whitespace both ends | `s.trim()` |
| Whitespace one end | `s.trim_start()` / `s.trim_end()` |
| Strip a character | `s.trim_matches('x')` |
| Replace all | `s.replace(a, b)` |
| Replace at most n | `s.replacen(a, b, n)` |

### Splitting and joining

| Need | Write |
|---|---|
| Split on a delimiter | `s.split(',')` (keeps empties) |
| Split on any whitespace | `s.split_whitespace()` (collapses) |
| At most n pieces | `s.splitn(n, ',')` / `s.rsplitn(n, ',')` |
| Keep the delimiter | `s.split_inclusive(',')` |
| Lines (handles CRLF) | `s.lines()` |
| Join | `slice.join(", ")` |
| Repeat | `"ab".repeat(3)` |

### Slicing

| Need | Write |
|---|---|
| Byte range, panics if not on boundary | `&s[a..b]` |
| Byte range, no panic | `s.get(a..b)` → `Option<&str>` |
| Boundary check | `s.is_char_boundary(i)` |
| Prefix by bytes | `&s[..n]` |

### Bytes and text

| Need | Write |
|---|---|
| `&str` → bytes | `s.as_bytes()` |
| `String` → bytes (consumes) | `s.into_bytes()` |
| Bytes → `&str` | `std::str::from_utf8(&v)` |
| Bytes → `Cow<str>`, lossy | `String::from_utf8_lossy(&v)` |

### Formatting

| Need | Write |
|---|---|
| Build a `String` | `format!("{x}")` |
| Append formatted to a `String` | `write!(s, "{x}")` (needs `use std::fmt::Write as _;`) |
| Align / pad | `{:>8}` `{:<8}` `{:^8}` `{:08}` |
| Sign / radix | `{:+}` `{:#x}` `{:#b}` |
| Truncate | `{:.3}` |
| Unambiguous output | `{:?}` |

### The diagnostics

| Code | Cause | Fix |
|---|---|---|
| `E0277` | `s[0]` | Range, or `.chars().nth(0)` |
| `E0308` | `String + String`, `push_str('c')` | `a + &b`; `push('c')` |
| `E0382` | Use after `+` moved the left operand | Reorder, or `clone()` |
| `E0505` | Move while borrowed | Drop the borrow first |
| `E0502` | Mutate while borrowed | End the borrow, or copy out |
| `E0106` | Returning `&str` from a local | Return `String` |
| runtime | Slice not on a char boundary | `get`, or check `is_char_boundary` |

### The one habit

Print `len()` and `chars().count()` together whenever a string surprises you. The pair tells you immediately whether you are looking at a byte problem or a character problem — which is the only question that matters in this chapter.


### What Comes Next

Chapter 04 ends with the same question it started with: who owns the bytes? A `String` owns them, a `&str` borrows them, and the compiler refuses to let the borrow outlive the owner. That rule has been visible in this notebook in three places — §3.4, §13, and every "returns `Option<&str>`" method — and it is the subject of the next chapter.

| Chapter | Title | What it settles |
|---|---|---|
| **05** | **Ownership and Borrowing** | Moves, copies, borrows, lifetimes, and why `&str` in / `String` out is the shape of nearly every function |
| 06 | Native Collections | `Vec`, `HashMap`, `HashSet`, `VecDeque`, `BTreeMap` — the containers a market-data path actually uses |
| 07 | I/O and Files | Reading, writing, buffering, and error propagation with `Result` |

If you want to feel how much of this chapter was actually about ownership, try to write a function that returns `&str` pointing into a local `String` you just created. The `E0106` you get is the whole of chapter 05 in one diagnostic.
